# 04 · Simulation studies

A small size-and-power study, followed by the results of two larger simulation studies read from
`studies/results/` (run `studies/study_exact.py` and `studies/study_phi.py` to reproduce them in full).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # or: pip install -e .  (from the repository root)
import numpy as np
import matplotlib.pyplot as plt
import gstarx_pw as g
from gstarx_pw.plotting import set_style, plot_links, plot_bootstrap, plot_heatmaps, OKABE_ITO as C
set_style()
import json, glob, pandas as pd

## 1. Size and power of the two-stage bootstrap test (N = 10, R = 100 per cell)

In [2]:
r0 = np.random.default_rng(7)
lat, lon = r0.uniform(-8.3, -6.7, 10), r0.uniform(111.0, 114.4, 10)
W0 = g.inverse_distance(lat, lon)
out = {}
for truth in ['prior', 'mixture', 'sparse']:
    for T in [100, 200]:
        rr = np.random.default_rng(T + len(truth)); M = g.annihilator(g.design_generic(T)); rej = 0
        for _ in range(100):
            Wt = {'prior': W0, 'sparse': g.sparse_matrix(10, rr), 'mixture': g.row_normalize(0.6 * W0 + 0.4 * g.sparse_matrix(10, rr))}[truth]
            A = np.diag(rr.uniform(0.45, 0.65, 10)) + rr.uniform(0.2, 0.4, 10)[:, None] * Wt
            rej += g.bootstrap_test(M @ g.simulate(A, T, rr), W0, M, rr, B=199)[0] <= 0.05
        out[(truth, T)] = rej / 100
pd.Series(out).unstack().round(2)

,100,200
mixture,0.11,0.14
prior,0.05,0.08
sparse,0.32,0.72


## 2. Size and power of the bootstrap test at N = 38, T = 156

In [3]:
E = {k: json.load(open(f'../studies/results/exact_{k}.json')) for k in ['gauss', 'garch', 't5', 'lag2a', 'lag2b', 'sparse', 'mixture']}
pd.DataFrame({v['design']: {'Ljung-Box rejections': v['lb12_share'], 'equation-wise rejection': v['eqwise_share'],
              'bootstrap rejection': v['boot_reject']} for v in E.values()}).T.round(3)

,Ljung-Box rejections,equation-wise rejection,bootstrap rejection
"H0, Gaussian",0.109,0.118,0.045
"H0, GARCH(1,1) common volatility",0.107,0.128,0.054
"H0, Student-t(5)",0.099,0.135,0.069
"H0, omitted lag-2 dynamics (c=0.40)",0.677,0.166,0.093
"H0, omitted lag-2 dynamics (c=0.55)",0.889,0.256,0.272
"H1 sparse, Gaussian",0.101,0.210,0.170
"H1 mixture, Gaussian",0.099,0.139,0.066


## 3. Weak spatial coefficients and starting values (N = 20, T = 200)

In [4]:
P = {c: json.load(open(f'../studies/results/phi_{c}.json')) for c in ['0.30', '0.20', '0.10', '0.05']}
pd.DataFrame({f'phi1 ~ {c}': {'prior': v['prior'], 'ratio (median)': v['raw_median'], 'VAR-implied': v['var'],
              'GSTARX-PW': v['pw'], 'median kappa': v['kappa_median'], 'sign flips': v['signflip'],
              'starts agree': v['agree']} for c, v in P.items()}).T.round(4)

,prior,ratio (median),VAR-implied,GSTARX-PW,median kappa,sign flips,starts agree
phi1 ~ 0.30,0.0883,0.2254,0.0587,0.0716,0.3,0.0100,0.9983
phi1 ~ 0.20,0.0885,0.5325,0.0733,0.0853,30.0,0.0567,0.9983
phi1 ~ 0.10,0.0892,1.2674,0.0874,0.0888,30.0,0.2017,1.0000
phi1 ~ 0.05,0.0890,1.3741,0.0937,0.0890,30.0,0.3800,1.0000
